# GSM8K paired model runs

Run from a repository checkout, using a GPU runtime in Colab. The default is 500 problems in each mode (1,000 responses). Raw pilot evidence remains in `outputs/raw/qwen3_1.7b_gsm8k.jsonl`. The shared parser and runner replace the pilot notebook?s overlapping repair cells.


In [ ]:
%pip install transformers accelerate datasets


In [ ]:
from pathlib import Path
import os

# In Colab, clone/upload the repository and set REPO_ROOT to that checkout.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "code/src/run_gsm8k.py").exists():
    REPO_ROOT = REPO_ROOT.parent.parent
assert (REPO_ROOT / "code/src/run_gsm8k.py").exists(), "Set REPO_ROOT to your repository checkout"
os.chdir(REPO_ROOT)


In [ ]:
NUM_PROBLEMS = 500
BATCH_SIZE = 1
OUTPUT = "outputs/raw/qwen3_1.7b_gsm8k_500.jsonl"

!python code/src/run_gsm8k.py --num-problems {NUM_PROBLEMS} --batch-size {BATCH_SIZE} --output {OUTPUT}


Rerun the generation cell to resume the same configuration. Choose a new output filename to change settings. Persist the output JSONL and its config JSON together across Colab sessions. Token limits are 8,192 for thinking and 2,048 for no-thinking.


In [ ]:
import json
from collections import defaultdict

rows = [json.loads(line) for line in Path(OUTPUT).read_text().splitlines()]
groups = defaultdict(list)
for row in rows:
    groups[row["mode"]].append(row)
for mode, values in groups.items():
    print(mode, {"saved": len(values), "expected": NUM_PROBLEMS,
        "accuracy_all": sum(r["correct"] for r in values) / len(values),
        "unparsed": sum(r["model_answer"] is None for r in values),
        "truncated": sum(r["is_truncated"] for r in values),
        "mean_tokens": sum(r["num_generated_tokens"] for r in values) / len(values)})
